In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

#パスの設定
input_path = Path("/content/drive/MyDrive/因果推論/h3_mesh_panel.csv")
input_path.parent.mkdir(parents=True, exist_ok=True)
output_dir = input_path.parent

#介入日
intervention_date = pd.Timestamp("2025-01-01")
#商圏重複率の除外基準
overlap_threshold = 0.60
#データの読み込む
df = pd.read_csv(input_path,parse_dates=["date"])

#必要な列が揃っているか確認
required_columns = {"date","h3_id","visitors","treated"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"必要な列がありません: {sorted(missing_columns)}")

#重複行ないか確認
if df.duplicated(["date", "h3_id"]).any():
    raise ValueError("date × h3_id に重複があります。")

#型を指定
df["treated"] = df["treated"].astype(int)
df["visitors"] = pd.to_numeric(df["visitors"],errors="raise")

#メッシュIDを確認
all_mesh_ids = sorted(df["h3_id"].unique())
treated_mesh_ids = sorted(df.loc[df["treated"] == 1,"h3_id"].unique())
control_mesh_ids = sorted(df.loc[df["treated"] == 0,"h3_id"].unique())
print("Number of all meshes:", len(all_mesh_ids))
print("Number of treated meshes:", len(treated_mesh_ids))
print("Number of control meshes:", len(control_mesh_ids))

#h3_id × station の対応表を作成
# h3_mesh_01 ～ h3_mesh_06 → shibuya
# h3_mesh_07 ～ h3_mesh_12 → ikebukuro
# h3_mesh_13 ～ h3_mesh_18 → ueno
# h3_mesh_19 ～ h3_mesh_24 → kawasaki
# h3_mesh_25 ～ h3_mesh_30 → omiya
# h3_mesh_31 ～ h3_mesh_36 → kichijoji

mesh_station_mapping = pd.DataFrame({
    "h3_id": [f"h3_mesh_{i:02d}" for i in range(1, 37)],
    "station": (["shibuya"] * 6 + ["ikebukuro"] * 6 + ["ueno"] * 6 + ["kawasaki"] * 6 + ["omiya"] * 6 + ["kichijoji"] * 6)})

#データフレームにstation列を追加
df = df.merge(mesh_station_mapping,on="h3_id",how="left",validate="many_to_one")

#整合性を確認
if df["station"].isna().any():
    raise ValueError("対応しないメッシュがあります。")

expected_treated = df["station"].eq("shibuya").astype(int)
if not np.array_equal(df["treated"].to_numpy(),expected_treated.to_numpy()):
    raise ValueError("stationとtreatedの対応が矛盾しています。")

#教材用に商圏重複スコアを定義（人工的に設定した値をJaccard係数として使用）
station_overlap_score = {
    "ikebukuro": 0.75,  #渋谷と池袋の来訪者集合のJaccard係数が0.75
    "ueno": 0.45,
    "kawasaki": 0.25,
    "omiya": 0.20,
    "kichijoji": 0.65,
}
#ドナーメッシュのメタデータ作成
donor_metadata = mesh_station_mapping.loc[mesh_station_mapping["station"] != "shibuya"].copy()
donor_metadata["overlap_score"] = donor_metadata["station"].map(station_overlap_score)
if donor_metadata["overlap_score"].isna().any():
    raise ValueError("overlap_scoreが設定されていない駅があります。")
donor_metadata["excluded_high_overlap"] = (donor_metadata["overlap_score"] >= overlap_threshold)

#除外するドナー
excluded_donors = donor_metadata.loc[donor_metadata["excluded_high_overlap"],"h3_id"].tolist()
#使用するドナー
eligible_donors = donor_metadata.loc[~donor_metadata["excluded_high_overlap"],"h3_id"].tolist()
#全てのドナー
all_donors = donor_metadata["h3_id"].tolist()

if len(eligible_donors) < 2:
    raise ValueError("除外後のドナー数が不足しています。")

#処置エリアの平均系列
#6つの渋谷メッシュの平均を1つの処置エリア系列とする。
treated_series = (df.loc[df["station"] == "shibuya"].groupby("date")["visitors"]
    .mean().sort_index())
treated_series.name = "actual_treated_area"

#ドナーメッシュを横持ちにする
donor_wide = (df.loc[df["station"] != "shibuya"].pivot(index="date",columns="h3_id",
        values="visitors").sort_index())
donor_wide = donor_wide.loc[:,all_donors]

#Pre / Post
pre_mask = treated_series.index < intervention_date
post_mask = treated_series.index >= intervention_date

#全てのドナーを使った場合の反実仮想と高重複ドナー除外後の仮想現実を作り比較するため、Synthetic Controlの関数を作成
def fit_scm(donor_units):
    #介入前の処置系列を作成
    y_pre = treated_series.loc[pre_mask].to_numpy(dtype=float)
    #介入前のドナー系列を作成
    x_pre = donor_wide.loc[pre_mask,donor_units].to_numpy(dtype=float)
    #エラーハンドリング
    if np.var(y_pre) == 0:
        raise ValueError("施策前の処置系列の分散が0です。")
    #重みを推定を目的とした最適化に代入する目的関数を定義
    def objective(weights):
        #期間全体で反実仮想を作成
        synthetic_pre = (x_pre @ weights)
        #観測値と反実仮想のGapを抽出
        residual = y_pre - synthetic_pre
        #mse/観測値の分散を計算
        return np.mean(residual ** 2) / np.var(y_pre)

    #最適化を実行し結果を格納
    #重みの初期値は全部同じ値になるように設定
    n_donors = len(donor_units)
    result = minimize(objective,x0=np.full(n_donors,1.0 / n_donors),
        method="SLSQP",bounds=[(0.0, 1.0) for _ in range(n_donors)],
        constraints={
            "type": "eq",
            "fun":lambda weights:weights.sum() - 1.0},
        options={
            "ftol": 1e-12,
            "maxiter": 5000,
            "disp": False,
        })
    #エラーハンドリング
    if not result.success:
        raise RuntimeError(f"SCM重み推定に失敗しました。\n {result.message}")
    #最適化で推定した重みを格納
    weights = pd.Series(result.x,index=donor_units,name="weight")
    #数値誤差レベルの重みを0へ
    weights[np.abs(weights) < 1e-8] = 0.0
    #反実仮想を作成
    synthetic = pd.Series(donor_wide[donor_units].to_numpy(dtype=float) @ weights.to_numpy(),
        index=donor_wide.index,name="synthetic")
    #観測値と反実仮想のGapを抽出
    gap = treated_series - synthetic
    pre_gap = gap.loc[pre_mask].to_numpy(dtype=float)
    post_gap = gap.loc[post_mask].to_numpy(dtype=float)
    #介入前後のRMSPEを計算
    pre_rmspe = np.sqrt(np.mean(pre_gap ** 2))
    post_rmspe = np.sqrt(np.mean(post_gap ** 2))
    #エラーハンドリング
    if np.isclose(pre_rmspe,0.0):
        raise ValueError("Pre RMSPEが0のためRatioを計算できません。")
    return {
        "weights": weights,
        "synthetic": synthetic,
        "gap": gap,
        "pre_rmspe": pre_rmspe,
        "post_rmspe": post_rmspe,
        "rmspe_ratio":post_rmspe / pre_rmspe,
        "mean_post_gap":gap.loc[post_mask].mean()
    }